# 🔬 เปรียบเทียบ 3 กลยุทธ์ Backtesting

**3 Scenarios:**
1. **Buy & Hold** - ซื้อแล้วถือ
2. **Rebalancing** - ปรับพอร์ตเป็นระยะ (Quarterly, Semi-Annual, Annual)
3. **DCA** - ลงทุนเป็นงวดๆ

---

## วิธีใช้:
1. แก้ password ใน Cell 1
2. Run Cell 1-5 ตามลำดับ
3. ดูผลเปรียบเทียบใน Cell 5

## Cell 1: Setup

In [ ]:
import sys
sys.path.append('/home/user/desktop-tutorial')

from backtesting.backtesting_engine import run_backtest
import backtesting.backtesting_engine as bt_engine
import mysql.connector
import pandas as pd
import numpy as np

# Database config
DB_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': 'krittanut123456',  # แก้ตรงนี้!
    'database': 'etf_backtesting'
}

bt_engine.DB_CONFIG = DB_CONFIG

# Test connection
try:
    conn = mysql.connector.connect(**DB_CONFIG)
    print('✅ MySQL Connected!')
    conn.close()
except Exception as e:
    print(f'❌ Connection failed: {e}')

## Cell 2: เลือก Portfolio

In [ ]:
# แสดง portfolios
conn = mysql.connector.connect(**DB_CONFIG)
df = pd.read_sql('SELECT portfolio_id, name FROM portfolios', conn)
conn.close()

print('Available Portfolios:')
print(df)

# เลือก portfolio
PORTFOLIO_ID = 1  # แก้ตรงนี้
print(f'\nSelected Portfolio ID: {PORTFOLIO_ID}')

## Cell 3: กำหนดพารามิเตอร์

In [ ]:
# Backtest parameters
START_DATE = '2020-01-01'
END_DATE = '2024-12-31'
INITIAL_CAPITAL = 100000
TRANSACTION_COST = 0.001

print(f'Period: {START_DATE} to {END_DATE}')
print(f'Capital: ${INITIAL_CAPITAL:,}')
print(f'Transaction Cost: {TRANSACTION_COST*100:.2f}%')

## Cell 4: Run All 3 Scenarios

In [ ]:
print('Running backtests...\n')

results = {}

# Scenario 1: Buy & Hold
print('1. Buy & Hold...')
try:
    bt_id = run_backtest(
        portfolio_id=PORTFOLIO_ID,
        start_date=START_DATE,
        end_date=END_DATE,
        initial_capital=INITIAL_CAPITAL,
        strategy_type='buy_hold',
        transaction_cost=TRANSACTION_COST
    )
    results['Buy & Hold'] = bt_id
    print(f'   ✅ Done (ID: {bt_id})')
except Exception as e:
    print(f'   ❌ Error: {e}')

# Scenario 2: Quarterly Rebalancing
print('\n2. Quarterly Rebalancing...')
try:
    bt_id = run_backtest(
        portfolio_id=PORTFOLIO_ID,
        start_date=START_DATE,
        end_date=END_DATE,
        initial_capital=INITIAL_CAPITAL,
        strategy_type='rebalancing',
        rebalance_frequency='quarterly',
        transaction_cost=TRANSACTION_COST
    )
    results['Rebalancing (Quarterly)'] = bt_id
    print(f'   ✅ Done (ID: {bt_id})')
except Exception as e:
    print(f'   ❌ Error: {e}')

# Scenario 3: Annual Rebalancing
print('\n3. Annual Rebalancing...')
try:
    bt_id = run_backtest(
        portfolio_id=PORTFOLIO_ID,
        start_date=START_DATE,
        end_date=END_DATE,
        initial_capital=INITIAL_CAPITAL,
        strategy_type='rebalancing',
        rebalance_frequency='annually',
        transaction_cost=TRANSACTION_COST
    )
    results['Rebalancing (Annual)'] = bt_id
    print(f'   ✅ Done (ID: {bt_id})')
except Exception as e:
    print(f'   ❌ Error: {e}')

print('\n✅ All backtests completed!')
print(f'Results: {results}')

## Cell 5: เปรียบเทียบผลลัพธ์

In [ ]:
# Get results from database
conn = mysql.connector.connect(**DB_CONFIG)

comparison = []

for strategy_name, bt_id in results.items():
    # Get backtest info
    df_info = pd.read_sql(f'''
        SELECT initial_capital, total_return
        FROM backtests
        WHERE backtest_id = {bt_id}
    ''', conn)
    
    # Get final value
    df_final = pd.read_sql(f'''
        SELECT portfolio_value, cumulative_return
        FROM backtest_results
        WHERE backtest_id = {bt_id}
        ORDER BY date DESC
        LIMIT 1
    ''', conn)
    
    # Get metrics
    df_returns = pd.read_sql(f'''
        SELECT daily_return
        FROM backtest_results
        WHERE backtest_id = {bt_id}
        ORDER BY date
    ''', conn)
    
    # Calculate metrics
    initial = df_info['initial_capital'].values[0]
    final = df_final['portfolio_value'].values[0]
    total_return = (final - initial) / initial * 100
    
    daily_returns = df_returns['daily_return'].values[1:]
    volatility = np.std(daily_returns) * np.sqrt(252) * 100
    
    days = len(df_returns)
    years = days / 252
    annual_return = ((final / initial) ** (1 / years) - 1) * 100
    
    sharpe = (annual_return - 2) / volatility if volatility > 0 else 0
    
    comparison.append({
        'Strategy': strategy_name,
        'Final Value': f'${final:,.0f}',
        'Total Return': f'{total_return:.2f}%',
        'Annual Return': f'{annual_return:.2f}%',
        'Volatility': f'{volatility:.2f}%',
        'Sharpe': f'{sharpe:.3f}'
    })

conn.close()

# Display comparison
df_comparison = pd.DataFrame(comparison)
print('\n' + '='*100)
print('📊 STRATEGY COMPARISON')
print('='*100)
print(f'\nPortfolio: {PORTFOLIO_ID}')
print(f'Period: {START_DATE} to {END_DATE}')
print(f'Initial Capital: ${INITIAL_CAPITAL:,}\n')
print(df_comparison.to_string(index=False))
print('\n' + '='*100)

# Find winner
best_return_idx = df_comparison['Total Return'].str.replace('%', '').astype(float).idxmax()
best_sharpe_idx = df_comparison['Sharpe'].astype(float).idxmax()

print('\n🏆 WINNERS:')
print(f'Best Return: {df_comparison.loc[best_return_idx, "Strategy"]}')
print(f'Best Sharpe: {df_comparison.loc[best_sharpe_idx, "Strategy"]}')
print('='*100)

---

## 📝 สรุป

ดูรายละเอียดเพิ่มเติมใน:
- `BACKTESTING_3_SCENARIOS.md` - คำอธิบายละเอียดทั้ง 3 กลยุทธ์
- `backtest_summary_*.txt` - รายงานแต่ละ backtest

**Happy Investing! 📊🚀**